<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "060"
iso_country = "Bermuda"
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2025
gdp = 9.190*1e9 # 2024

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Bermuda': 

Number of trade records present in dataset: 13837
Number of partner countries present in dataset: 136
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$889.5m
Total weight within the dataset: 740.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,060,Bermuda,232,Eritrea,Import,Sea,620469,"620469 | Trousers, bib and br",6204,"6204 | Suits, ensembles, ja",62,62 | Apparel and clothing,28233,"Women's or girls' suits, coats, jackets, dress...",Textiles and Wearing Apparel,28,WAP - Wearing apparel,23.571369,23.571369,23.571369,u,0.6221,0.671868,0.813212,11816.0,7342.119736,5.970702,['Container'],['Container'],0.031208,0.031153,0.031159,0.031159,0.031183,0.031183,0.032645,0.057857,0.036008,0.033431,0.033521,0.034361,0.031965,0.056662,0.037901,0.034717,0.033920,0.035456,0.031369,0.061616,0.046999,0.041104,0.038029,0.036956,0.030784,0.065513,0.053910,0.050518,0.040715,0.039608,0.030210,0.066389,0.063428,0.063974,0.044139,0.041607,0.029624,0.065024,0.062313,0.068273,0.050501,0.042369,0.029072,0.058003,0.060455,0.063350,0.047489,0.04287,0.02851,0.053281,0.054161,0.067559,0.055813,0.043154,28233
1,1,060,Bermuda,232,Eritrea,Import,Sea,640411,640411 | Sports footwear; ten,6404,6404 | Footwear; with outer,64,64 | Footwear; gaiters an,2942,"Tennis shoes, basketball shoes, gym shoes, tra...",Textiles and Wearing Apparel,29,LEA - Leather products,88.632230,88.632230,88.632230,2u,1.5242,1.668999,3.403478,11816.0,7342.119736,24.988740,['Container'],['Container'],0.130612,0.130384,0.130406,0.130406,0.130508,0.130508,0.136627,0.242145,0.150704,0.139914,0.140294,0.143808,0.133782,0.237144,0.158625,0.145298,0.141964,0.148393,0.131286,0.257878,0.196703,0.172029,0.159160,0.154670,0.128837,0.274188,0.225625,0.211430,0.170401,0.165767,0.126434,0.277851,0.265462,0.267744,0.184732,0.174133,0.123983,0.272140,0.260794,0.285738,0.211358,0.177325,0.121674,0.242756,0.253016,0.265134,0.198753,0.17942,0.11932,0.222992,0.226677,0.282750,0.233591,0.180608,2942


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $248.5m (27.9%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
22 | Beverages, spirits a  51536059.6  262212459.2      5.8     35.4
19 | Preparations of cere  24878733.2    8877434.2      2.8      1.2
02 | Meat and edible meat  23384235.5    8226272.6      2.6      1.1
21 | Miscellaneous edible  22211264.1    6689548.4      2.5      0.9
04 | Dairy produce; birds  20801711.4    9136691.9      2.3      1.2
08 | Fruit and nuts, edib  17508104.0   12575825.3      2.0      1.7
16 | Meat, fish, crustace  14654731.4    3244118.0      1.6      0.4
20 | Preparations of vege  14463241.2    9385314.1      1.6      1.3
07 | Vegetables and certa  13072049.6   14126741.3      1.5      1.9
03 | Fish and crustaceans  12357276.8    2016175.3      1.4      0.3
23 | Food industries, res   7843999.0    5854173.4      0.9      0.8
09

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $225.2m (25.3%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  143095793.6  297055516.7     16.1     40.1
Agriculture        77646875.0   48346093.8      8.7      6.5
Fishing             4454263.3     669920.3      0.5      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $175.1m (19.7%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     91026365.4  39668748.0     10.2      5.4
OMT - Meat products n.e.c.     25619844.0   9254038.1      2.9      1.3
V_F - Vegetables, fruit, nuts  23824657.9  21699792.9      2.7      2.9
MIL - Dairy products           14885777.1   4312624.1      1.7      0.6
CMT - Bovine meat prods        10139090.2   1733869.0      1.1      0.2
OCR - Crops n.e.c.              3222768.7   2772796.0      0.4      0.4
VOL - Vegetable oils and fats   3031394.3   1995034.1      0.3      0.3
PCR - Processed rice            1265991.1   2199026.5      0.1      0.3
SGR - Sugar                     1015153.4   2319976.7      0.1      0.3
OAP - Animal products n.e.c.     621314.9    282990.0      0.1      0.0
FSH - Fishing                    162061.3     34147.6      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 27.9% 
GTAP 19.7% 
EORA 25.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23066/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,060,Bermuda,232,Eritrea,Import,Sea,620469,"620469 | Trousers, bib and br",6204,"6204 | Suits, ensembles, ja",62,62 | Apparel and clothing,28233,"Women's or girls' suits, coats, jackets, dress...",Textiles and Wearing Apparel,28,WAP - Wearing apparel,23.571369,23.571369,23.571369,u,0.6221,0.671868,0.813212,11816.0,7342.119736,5.970702,['Container'],['Container'],0.031208,0.031153,0.031159,0.031159,0.031183,0.031183,0.032645,0.057857,0.036008,0.033431,0.033521,0.034361,0.031965,0.056662,0.037901,0.034717,0.033920,0.035456,0.031369,0.061616,0.046999,0.041104,0.038029,0.036956,0.030784,0.065513,0.053910,0.050518,0.040715,0.039608,0.030210,0.066389,0.063428,0.063974,0.044139,0.041607,0.029624,0.065024,0.062313,0.068273,0.050501,0.042369,0.029072,0.058003,0.060455,0.063350,0.047489,0.04287,0.02851,0.053281,0.054161,0.067559,0.055813,0.043154,28233,NaN
1,1,060,Bermuda,232,Eritrea,Import,Sea,640411,640411 | Sports footwear; ten,6404,6404 | Footwear; with outer,64,64 | Footwear; gaiters an,2942,"Tennis shoes, basketball shoes, gym shoes, tra...",Textiles and Wearing Apparel,29,LEA - Leather products,88.632230,88.632230,88.632230,2u,1.5242,1.668999,3.403478,11816.0,7342.119736,24.988740,['Container'],['Container'],0.130612,0.130384,0.130406,0.130406,0.130508,0.130508,0.136627,0.242145,0.150704,0.139914,0.140294,0.143808,0.133782,0.237144,0.158625,0.145298,0.141964,0.148393,0.131286,0.257878,0.196703,0.172029,0.159160,0.154670,0.128837,0.274188,0.225625,0.211430,0.170401,0.165767,0.126434,0.277851,0.265462,0.267744,0.184732,0.174133,0.123983,0.272140,0.260794,0.285738,0.211358,0.177325,0.121674,0.242756,0.253016,0.265134,0.198753,0.17942,0.11932,0.222992,0.226677,0.282750,0.233591,0.180608,2942,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $132.0m of $889.5m (14.8%)
Volume: 72.6 of 740.0 million kilograms (9.8%)
Value-to-Weight: 1.82 from 1.2 US$ per kg (+51.4%)
Average distance to market (volume-weighted): 2022.8 from 2215.0 nm (+-8.7%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    287  53559158.4  21772782.9      6.0      2.9
1        Meat and Prep     70  19577558.5   7397973.4      2.2      1.0
2           Vegetables    186  17099083.5  17395848.8      1.9      2.4
3               Fruits    122  16445388.7  12089345.6      1.8      1.6
4                Dairy     61  16008951.3   6682763.3      1.8      0.9
5      Sugar and Honey     71   5115642.5   4027300.8      0.6      0.5
6        Fats and Oils     72   3951343.1   2217059.6      0.4      0.3
7  Unprocessed Cereals     31    287445.4    997023.5      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,7559011.9,13084979.2,8235072.3,7685194.4,7706240.0,7919995.6,7028224.9,14903240.4,14176970.2,14484552.7,10023091.3,9484932.6,6655144.1,12046114.4,12275684.8,15019338.2,12302691.4,9825383.8
1,Fats and Oils,2435964.8,4193112.9,2652098.2,2474104.2,2480855.7,2551225.8,2266506.4,4774177.7,4557040.7,4650362.4,3225790.8,3050372.0,2147735.6,3871288.4,3946314.0,4817899.6,3962566.4,3157933.6
2,Fruits,12994427.8,22813620.2,14262284.6,13268519.4,13304674.8,13652317.0,12047966.7,26100714.0,24895160.9,25237094.7,17433342.2,16458337.6,11385566.5,21006969.7,21375256.1,26460141.0,21787933.3,17061718.8
3,Meat and Prep,9117353.2,16158810.3,10056732.8,9336740.6,9362097.1,9596578.7,8437187.4,18541515.0,17714776.9,17867089.9,12327524.0,11620242.0,7962414.3,14880652.6,15126572.6,18868447.1,15587984.9,12052325.0
4,Processed Cereals,23542575.6,40090606.0,25457384.8,23831532.2,23897813.2,24599514.8,21954636.3,45655323.0,43358494.2,44614311.8,30938867.6,29247644.6,20835607.7,37065649.7,37844140.3,45652382.8,37337934.9,30260498.5
5,Sugar and Honey,4333626.6,7341879.2,4678332.0,4381212.9,4393374.9,4525029.3,4044644.1,8348198.5,7936631.8,8174474.5,5676941.0,5369730.0,3841060.9,6793395.4,6938707.2,8344657.6,6829177.2,5553573.3
6,Unprocessed Cereals,748497.4,1218976.0,796718.4,750707.2,752956.5,777643.3,702860.9,1416440.2,1346980.1,1401065.4,977399.7,908209.1,670790.3,1160863.5,1192010.2,1381536.1,1146699.3,935307.7
7,Vegetables,18974246.4,33099540.2,20753887.4,19336020.2,19389078.0,19909540.6,17615327.2,37843309.4,36052026.3,36663229.8,25328484.4,23932771.7,16659964.0,30493915.8,31051121.5,38208112.3,31405185.0,24800263.7


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.35,0.04,0.01,0.01,0.02,0.49,0.45,0.47,0.19,0.15,0.34,0.35,0.52,0.35,0.20
1,Fats and Oils,0.44,0.05,0.01,0.01,0.03,0.63,0.58,0.60,0.24,0.20,0.44,0.46,0.68,0.46,0.26
2,Fruits,0.60,0.08,0.02,0.02,0.04,0.85,0.78,0.80,0.33,0.27,0.59,0.61,0.92,0.63,0.35
3,Meat and Prep,0.36,0.05,0.01,0.01,0.02,0.52,0.47,0.48,0.20,0.16,0.35,0.37,0.56,0.39,0.21
4,Processed Cereals,0.31,0.04,0.01,0.01,0.02,0.44,0.40,0.42,0.17,0.14,0.30,0.32,0.46,0.31,0.18
5,Sugar and Honey,0.59,0.07,0.01,0.01,0.04,0.84,0.76,0.81,0.32,0.26,0.58,0.61,0.88,0.58,0.33
6,Unprocessed Cereals,1.64,0.17,0.01,0.02,0.10,2.48,2.24,2.43,0.96,0.71,1.70,1.81,2.47,1.66,0.92
7,Vegetables,0.83,0.10,0.02,0.02,0.05,1.18,1.08,1.11,0.45,0.37,0.81,0.84,1.26,0.86,0.48
